# Saving Enriched Data into GOLD Layer and save as delta table.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window as W

In [0]:
%skip
%sql
-- create schema if not exists onehouse_dev_dtu_gold.gold_final_schema;

In [0]:
# Step 1: Read Silver tables
orders = spark.table("onehouse_dev_dtu_silver.silver_cleaned_schema.silver_orders").alias("o")
customers = spark.table("onehouse_dev_dtu_silver.silver_cleaned_schema.silver_customers").alias("c")
products = spark.table("onehouse_dev_dtu_silver.silver_cleaned_schema.silver_products").alias("p")
regions = spark.table("onehouse_dev_dtu_silver.silver_cleaned_schema.silver_regions").alias("r")

# Step 2: Join all 4 tables
gold_sales_report = (
    orders
    .join(
        customers,
        F.col("o.customer_id") == F.col("c.customer_id"),
        "inner"
    )
    .join(
        products,
        F.col("o.product_id") == F.col("p.product_id"),
        "inner"
    )
    .join(
        regions,
        F.col("c.region") == F.col("r.region"),
        "left"
    )

    # Step 3: Select report columns and calculate sales
    .select(
        F.col("o.order_id").alias("order_id"),
        F.col("o.order_date").alias("order_date"),
        F.col("o.customer_id").alias("customer_id"),
        F.col("c.customer_name").alias("customer_name"),
        F.col("c.region").alias("region"),
        F.col("r.region_manager").alias("region_manager"),
        F.col("o.product_id").alias("product_id"),
        F.col("p.product_name").alias("product_name"),
        F.col("p.category").alias("category"),
        F.col("o.quantity").alias("quantity"),
        F.col("p.unit_price").alias("unit_price"),
        (
            F.col("o.quantity") * F.col("p.unit_price")
        ).cast("decimal(18,2)").alias("gross_sales"),
        F.col("o.status").alias("status")
    )
)

# Step 4: Save as Gold Delta table
(
    gold_sales_report.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("onehouse_dev_dtu_gold.gold_final_schema.gold_sales_report")
)

print("Gold sales report created successfully!")

In [0]:
display(
    spark.table("onehouse_dev_dtu_gold.gold_final_schema.gold_sales_report")
    .orderBy("order_id")
)

In [0]:
display(
    spark.table("onehouse_dev_dtu_gold.gold_final_schema.gold_sales_report")
    .groupBy("status")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("gross_sales").alias("total_sales")
    )
)

Why these joins?
Join	Purpose
Orders INNER JOIN Customers	Get customer details
Orders INNER JOIN Products	Get product details and price
Customers LEFT JOIN Regions	Get the regional manager while retaining customers with missing region mappings

Your final Gold table contains one row per order, enriched with customer, product, and regional details.

Optimization note: For your small customer, product, and region lookup tables, broadcast joins may improve performance when the orders table is large. Add F.broadcast(customers) or F.broadcast(products) only when those tables are small enough to fit safely in executor memory. Also, use overwrite mode for this practice project; production pipelines generally need controlled incremental loads or MERGE logic.